In [ ]:
import os
import sqlite3
import random
import zipfile
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from google.colab import files

In [ ]:
# Libreria Fpdf per la generazione del pdf della policy
try:
  from fpdf import FPDF
except ImportError:
  !pip install fpdf2
  from fpdf import FPDF

print("=== INIZIO GENERAZIONE ASSET PER NOVABANCA RISK & CREDIT HUB ===")

=== INIZIO GENERAZIONE ASSET PER NOVABANCA RISK & CREDIT HUB ===


In [ ]:
# Fix dei seed per riproducibilità
np.random.seed(42)
random.seed(42)

In [ ]:
# Directory temporanea di lavoro
OUTPUT_DIR = "generated_assets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ----------------------------------------------------------------------
# 1. CREAZIONE DEL DATABASE SQLITE CON 4 TABELLE E DATI SPORCHI (10-12%)
# ----------------------------------------------------------------------

db_path = os.path.join(OUTPUT_DIR, "novabanca_core_banking.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Abilita vincoli di foreign key
cursor.execute("PRAGMA foreign_keys = ON; ")

# Creazione tabelle
cursor.execute("""
CREATE TABLE IF NOT EXISTS T_FILIALI(
  Filiale_ID TEXT PRIMARY KEY,
  Nome_Filiale TEXT NOT NULL,
  Area_Geografica TEXT NOT NULL,
  Direttore_Area TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS T_CLIENTI (
  ID_Cliente TEXT PRIMARY KEY,
  Filiale_ID TEXT NOT NULL,
  Eta_Cliente REAL,
  Categoria_Prof TEXT,
  Reddito_Annuale_EUR TEXT,
  FOREIGN KEY (Filiale_ID) REFERENCES T_FILIALI (Filiale_ID)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS T_PRATICHE_FIDO (
  ID_Pratica TEXT PRIMARY KEY,
  ID_Cliente TEXT NOT NULL,
  Importo_Richiesto_EUR REAL,
  Credit_Score_Interno INTEGER,
  Stato_Pratica TEXT NOT NULL,
  FOREIGN KEY (ID_Cliente) REFERENCES T_CLIENTI (ID_Cliente)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS T_PERFORMANCE_AMORT (
  ID_Pagamento TEXT PRIMARY KEY,
  ID_Pratica TEXT NOT NULL,
  Rata_Mensile_EUR REAL,
  Giorni_Ritardo_Pagamento INTEGER,
  Flag_Default_12M INTEGER,
  FOREIGN KEY (ID_Pratica) REFERENCES T_PRATICHE_FIDO (ID_Pratica)
);
""")

In [ ]:
# --- Populate T_FILIALI ---
filiali_data = [
    ("FIL_001", "Milano Hub Central", "Nord", "Marco Rossi"),
    ("FIL_002", "Torino San Paolo", "Nord", "Elena Bianchi"),
    ("FIL_003", "Bologna Marconi", "Nord", "Stefano Ferrari"),
    ("FIL_004", "Roma EUR Core", "Centro", "Giulia Romano"),
    ("FIL_005", "Firenze Signoria", "Centro", "Alessandro Conti"),
    ("FIL_006", "Napoli Toledo", "Sud", "Antonio Esposito"),
    ("FIL_007", "Bari Aldo Moro", "Sud", "Maria Russo"),
    ("FIL_008", "Palermo Liberty", "Isole", "Giuseppe Marino")
]
cursor.executemany("INSERT INTO T_FILIALI VALUES (?, ?, ?, ?)", filiali_data)

In [ ]:
# --- Generation of Raw Data (> 1000 records) ---
N_RECORDS = 1200

categorie_prof = [
    "Dipendente Tempo Indeterminato",
    "Dipendente Tempo Determinato",
    "Libero Professionista",
    "Imprenditore",
    "Pensionato",
    "Dirigente"
]

stati_pratica = ["Approvata","In Valutazione","Respinta","Rinegoziata"]

clienti_list = []
pratiche_list = []
amort_list = []

for i in range(1, N_RECORDS + 1):
  id_cliente = f"CLI_{i:04d}"
  id_pratica = f"PRA_{i:04d}"
  id_pagamento = f"PAG_{i:04d}"
  filiale_id = random.choice([f[0] for f in filiali_data])

  # 1. Generazione dati clienti con sporcizia
  eta = float(random.randint(21, 72))
  cat_prof = random.choice(categorie_prof)
  reddito_num = float(random.randint(18000, 115000))
  reddito_str = f"{reddito_num:,.0f} €".replace(",",".")

  # Inserimento sporcizia (10-12% dei dati)
  dirty_type = random.random()
  if dirty_type < 0.04:
    # Reddito Nullo / NaN
    reddito_str = None
  elif dirty_type < 0.08:
    # Età Nulla / NaN
    eta = None
  elif dirty_type < 0.11:
    # Spazi sporchi nella categoria professionale
    cat_prof = f" {cat_prof} "

  clienti_list.append((id_cliente, filiale_id, eta, cat_prof, reddito_str))

  # 2. Generazione dati Pratica Fido (con sporcizia)
  importo = round(random.uniform(5000.0, 350000.0),2)
  score = random.randint(320,840)
  stato = random.choice(stati_pratica)

  # Outlier nel credit score (3% delle pratiche)
  if random.random() < 0.03:
    score = random.randint(910, 999) # Fuori dalla scala valida di valori

  pratiche_list.append((id_pratica, id_cliente, importo, score, stato))

  # 3. Generazione dati ammortamento & performance
  rata = round(importo / random.randint(24, 120),2)

  # Correlazione con ritardi e default
  if score < 550:
    ritardo = random.choice([0, 15, 30, 60, 95, 120])
  else:
    ritardo = random.choice([0, 0, 0, 0, 5, 12, 25])

  flag_default = 1 if ritardo >= 90 else 0

  amort_list.append((id_pagamento, id_pratica, rata, ritardo, flag_default))

cursor.executemany("INSERT INTO T_CLIENTI VALUES (?, ?, ?, ?, ?)", clienti_list)
cursor.executemany("INSERT INTO T_PRATICHE_FIDO VALUES (?, ?, ?, ?, ?)", pratiche_list)
cursor.executemany("INSERT INTO T_PERFORMANCE_AMORT VALUES (?, ?, ?, ?, ?)", amort_list)

conn.commit()
conn.close()

print(f"-> DATABASE SQLite creato con successo: {db_path}")
print(f" Totale Clienti: {len(clienti_list)}, Pratiche: {len(pratiche_list)}")

-> DATABASE SQLite creato con successo: generated_assets/novabanca_core_banking.db
 Totale Clienti: 1200, Pratiche: 1200


In [ ]:
# ----------------------------------------------------------------------
# 2. CREAZIONE DOCUMENTI DELLA KNOWLEDGE BASE RAG
# ----------------------------------------------------------------------

kb_dir = os.path.join(OUTPUT_DIR, "knowledge_base")
os.makedirs(kb_dir, exist_ok=True)

In [ ]:
# 2.1 PDF: Policy Erogazione Credito 2026
class PDFPolicy(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 12)
        self.cell(0, 10, 'NOVABANCA - RISK MANAGEMENT POLICY 2026', 0, 1, 'C')
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Pagina {self.page_no()}', 0, 0, 'C')

pdf = PDFPolicy()
pdf.add_page()
pdf.set_font("Arial", 'B', 14)
pdf.cell(0, 10, "Policy Ufficiale di Erogazione Credito e Fidi", 0, 1, 'L')
pdf.ln(5)

pdf.set_font("Arial", '', 10)
policy_text = """
1. AMBITO DI APPLICAZIONE ED OBIETTIVI
La presente Policy stabilisce i criteri direttivi per la concessione del credito a clienti Private e Corporate del Gruppo NovaBanca per l'anno finanziario 2026.

2. PARAMETRI FONDAMENTALI DI VALUTAZIONE
- Debt-to-Income Ratio (DTI): Il rapporto tra il totale dei debiti mensili del richiedente e il reddito netto mensile non deve superare la soglia massima del 35% per redditi sotto i 30.000 EUR e del 40% per redditi superiori.
- Loan-to-Value Ratio (LTV): Per le pratiche di mutuo ipotecario residenziale, il valore finanziato (LTV) non puo eccedere l'80% del valore di perizia dell'immobile.
- Credit Score Interno (NVB Score): Il punteggio minimo ammissibile per la delibera automatica e fissato a 620 punti (scala 300-850).

3. PROCESSO DI ESALATION E APPROVAZIONE
- Pratiche con Credit Score tra 550 e 619 richiedono la firma congiunta del Direttore di Filiale e del Risk Officer d'Area.
- Pratiche con Credit Score inferiore a 550 o DTI superiore al 45% sono rigettate automaticamente dal sistema centrale.
"""
pdf.multi_cell(0, 6, policy_text)
pdf_path = os.path.join(kb_dir, "Policy_Erogazione_Credito_2026.pdf")
pdf.output(pdf_path)

# 2.2 TXT: Manuale Gestione NPL e Crediti Deteriorati
npl_text = """= NOVABANCA - MANUALE OPERATIVO GESTIONE NPL E CREDITI DETERIORATI =

1. CLASSIFICAZIONE DEGLI EXPOSURES
I crediti deteriorati si dividono in tre categorie principali secondo la normativa EBA e le linee guida Banca d'Italia:
a) Esposizioni Scadute e/o Sconfinanti Deteriorate (Past Due): Crediti con ritardo nei pagamenti continuativo da oltre 90 giorni.
b) Inadempienze Probabili (Unlikely to Pay - UTP): Crediti per i quali si ritiene improbabile il recupero totale senza il ricorso ad azioni come la escussione delle garanzie.
c) Sofferenze (Bad Loans): Esposizioni verso soggetti in stato di insolvenza o in situazioni equiparabili.

2. STRATEGIE DI RECOVERY E RISTRUTTURAZIONE
- Entro 30 giorni dal primo ritardo: Invio primo sollecito automatico via PEC / SMS.
- Entro 60 giorni dal primo ritardo: Contatto telefonico da parte del Credit Recovery Officer.
- Superati i 90 giorni (Flag_Default_12M = 1): Trasferimento automatico della pratica al Dipartimento Non-Performing Loans (NPL). Si valuta il piano di rientro straordinario o l'avvio della procedura giudiziale.
"""
txt_path = os.path.join(kb_dir, "Manuale_Gestione_NPL_e_Crediti_Deteriorati.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(npl_text)

# 2.3 MD: FAQ Operative Consulenti Risk
faq_text = """# FAQ Operative - Consulenti Risk & Credit NovaBanca

### Q1: Qual e il tasso soglia usura attualmente in vigore per i finanziamenti personali?
**R:** Ai sensi della legge 108/96 e secondo le rilevazioni trimestrali di Banca d'Italia, il Tasso Effettivo Globale Medio (TEGM) viene aggiornato con cadenza trimestrale. Per il trimestre corrente, la soglia di usura e calcolata aumentando il TEGM del 25% a cui si aggiungono ulteriori 4 punti percentuali.

### Q2: Come ci si comporta in caso di discrepanze nelle categorie professionali o dati mancanti nei form di richiesta?
**R:** Il Data Cleaning Engine del nostro Data Agent pulisce automaticamente le stringhe grezze, convertendo i redditi formattati (es. "45.000 €") in float numerici e rimuovendo spazi superflui. In presenza di valori mancanti (NaN) su campioni trascurabili (<5%), le pratiche vengono escluse dal calcolo delle medie per non distorcere le analisi di rischio.

### Q3: Quali requisiti impone la direttiva MiFID II per la vendita di prodotti finanziari abbinati al credito?
**R:** E obbligatoria la profilatura adeguata del cliente tramite questionario MiFID II prima dell'emissione di qualsiasi prodotto finanziario complesso abbinato al fido.
"""
md_path = os.path.join(kb_dir, "FAQ_Operative_Consulenti_Risk.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write(faq_text)

print(f"-> Knowledge Base creata con successo in: {kb_dir}")

-> Knowledge Base creata con successo in: generated_assets/knowledge_base


/tmp/ipykernel_2662/4097594913.py:4: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  self.set_font('Arial', 'B', 12)
/tmp/ipykernel_2662/4097594913.py:5: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, 'NOVABANCA - RISK MANAGEMENT POLICY 2026', 0, 1, 'C')
/tmp/ipykernel_2662/4097594913.py:15: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", 'B', 14)
/tmp/ipykernel_2662/4097594913.py:16: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Policy Ufficiale di Erogazione Credito e Fidi", 0, 1, 'L')
/tmp/ipykernel_2662/4097594913.py:19: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated sinc

In [ ]:
# ----------------------------------------------------------------------
# 3. COMPRESSIONE ZIP E DOWNLOAD AUTOMATICO
# ----------------------------------------------------------------------

zip_filename = "novabanca_asset_passo1.zip"
zip_path = os.path.join("/content", zip_filename)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
  # Aggiungi DB
  zipf.write(db_path, arcname="novabanca_core_banking.db")
  # Aggiungi KB
  for root, _, files_list in os.walk(kb_dir):
    for file in files_list:
      full_file_path = os.path.join(root, file)
      arcname = os.path.join("knowledge_base", file)
      zipf.write(full_file_path, arcname=arcname)

print(f"\n=== COMPLETATO! Scaricamento del file '{zip_filename}' in corso... ===")
files.download(zip_path)


=== COMPLETATO! Scaricamento del file 'novabanca_asset_passo1.zip' in corso... ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>